# Creating Vanilla Rainbow Tables

For N = 2**16, currently only making one table

## Imports

In [11]:
import pickle
import random
from hashlib import sha256
from tqdm import tqdm
from math import pi, sqrt, e, log

## Table Parameters

In [12]:
# initialise startpoints - either generate them or load from pickle - to keep same across runs
def get_startpoints(N, m_0, nlabel, alpha):
    # try opening pickle file, else generate and save
    try:
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'rb') as f:
            startpoints = pickle.load(f)

    # no file found - generate and save
    except FileNotFoundError:
        # random but unique - store as a set?
        startpoints = set()
        while len(startpoints) < m_0:
            startpoints.add(random.randint(0, N-1))

        # store startpoints in pickle file
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'wb') as f:
            pickle.dump(startpoints, f)

    # return the startpoints
    return startpoints

In [13]:
# label N to find easier - label is the exponent
nlabel = 16
##############################################################################
N = 2 ** 16 # keyspace
p = 1 - e ** -2 # our table coverage - 86%
##############################################################################
t = round(log(1-p)/log(1-N**(-1/3))) # chain length t
alpha = 0.95 # maximality factor
mt_target = N**(2/3) # our target mt
m_0 = round(mt_target/(1-alpha))    # m_0 - number of startpoints
##############################################################################
# initialise startpoints 
startpoints = get_startpoints(N, m_0, nlabel, alpha)


## Hash and Reduction Functions

In [14]:
# Hash function
def H(x):
	return int(sha256(bytes(x)).hexdigest(), 16)

# Reduction function
# currently mod but should change to murmurhash in future
def r(y, i, ell=0):   # also takes in ell - number of tables - for future use (but currently ell=0)
	return (y + i + ell*t) % N

## Building Vanilla Table

In [15]:
# take in m_0 and t as parameters - how many chains to start with and how long to make the chains
# store the table as a dictionary of endpoint:startpoint pairs (rather than sp:ep for easier lookup later)
# store in a pickle file

def build_vanilla_table(m_0, t, alpha, startpoints):
    # store table in dictionary
    table = {}

    # for each startpoint
    for j in tqdm(range(m_0)):
        # pop the next startpoint off
        sp = startpoints.pop()
        current_point = sp  # keep track of current point in chain -  we want to store startpoint later

        # create the chain
        for i in range(t):
            # hash then reduce the value
            current_point = r(H(current_point), i)

        # check if there wasn't a chain merge (not in a value stored already) - if not then store in table
        if current_point not in table.keys():
            table[current_point] = sp

    # store the table as a pickle file
    with open(f'vanilla_table_alpha_{alpha}_t_{t}.pkl', 'wb') as f:
        pickle.dump(table, f)

    return table

## Searching the Table

In [16]:
def search_vanilla_table(y, t, table):
    # keep track of column we're in 
    c = t - 1
    # reduce (r_t-1) the hash then compare in the table
    x = r(y, c)

    # while we haven't reached the end of our chain
    while c > 0:
        # if there is a match in the keys (our endpoints), regenerate chain until we find the key
        try:
            point = table[x]   # search for endpoint in table and get startpoint
            # regenerate chain until column c - we are now in the column before the match
            for i in range(c):
                point = r(H(point), i)
            
            # hash the point - if it is a match we have found our preimage 
            if H(point) == y:
                return point
            
            # if that didn't work then we ran into a false alarm
 
        # if we didn't find a match in endpoints, we need to restart the search
        except:
            # reduce c by 1 to move to the previous column
            c -= 1
            # number of columns between current colum and end 
            diff = t - c  
            # reduce y by the new c index
            x = r(y, c)
            # then hash and reduce however many times to move back through columns
            for i in range(diff - 1, 0, -1):    # decrease difference by 1 (as we already reduced by current diff index) then continuously reduce by 1
                x = r(H(x), t-i)

    # We have searched all columns (or there was a false alarm) - return -1
    return -1

## Run

### Precomputation Phase - Build the Table

In [17]:
# either build or load table
def get_vanilla_table():
    # try loading table from pickle file
    try:
        with open(f'vanilla_table_alpha_{alpha}_t_{t}.pkl', 'rb') as f:
            table = pickle.load(f)

    # if no pickle file found, build the table
    except FileNotFoundError:
        table = build_vanilla_table(m_0, t, alpha, startpoints)

    return table

In [18]:
vanilla = get_vanilla_table()

### Online Phase - Using the Table

In [ ]:
# test search gives true positive
y = H(53780)
search_vanilla_table(y, t, vanilla)
# test search gives true negative
# test search gives -1 for false alarm

In [20]:
vanilla

{42934: 0,
 54156: 1,
 53948: 2,
 395: 5,
 47073: 6,
 2793: 9,
 14381: 10,
 46844: 14,
 31258: 15,
 55518: 16,
 12004: 22,
 42210: 24,
 2456: 25,
 19350: 27,
 24775: 28,
 47890: 29,
 14002: 30,
 542: 31,
 6315: 33,
 59914: 36,
 28694: 38,
 34495: 39,
 53917: 41,
 28689: 42,
 1369: 45,
 58397: 48,
 60996: 49,
 50948: 50,
 13624: 59,
 13849: 61,
 1382: 62,
 46814: 64,
 34797: 65,
 6766: 67,
 48074: 68,
 32557: 71,
 59105: 78,
 22686: 80,
 17567: 84,
 2583: 85,
 45558: 86,
 63591: 87,
 30353: 88,
 1477: 90,
 38979: 91,
 54805: 92,
 22718: 94,
 2489: 95,
 63140: 96,
 511: 98,
 44675: 99,
 2894: 102,
 42008: 105,
 42759: 107,
 64130: 108,
 2470: 111,
 55601: 112,
 19898: 117,
 10461: 120,
 59793: 124,
 64321: 128,
 15706: 130,
 46561: 133,
 62605: 141,
 36056: 143,
 48291: 144,
 45557: 146,
 44885: 148,
 6287: 149,
 53304: 152,
 28091: 153,
 64767: 155,
 52695: 156,
 63247: 161,
 37052: 164,
 53832: 165,
 64798: 167,
 34796: 169,
 46867: 170,
 11304: 171,
 28162: 172,
 56615: 173,
 44683: 1

In [21]:
# get chain from pickle file
with open(f'vanilla_chain_alpha_{alpha}_t_{t}.pkl', 'rb') as f:
    chain = pickle.load(f)

chain

[0,
 47189,
 29025,
 60304,
 46135,
 4587,
 42540,
 39845,
 48391,
 5761,
 38945,
 38833,
 46625,
 58943,
 53780,
 25319,
 53601,
 63816,
 29971,
 49893,
 60068,
 38506,
 32561,
 30958,
 31185,
 30898,
 49680,
 61178,
 46304,
 63310,
 17066,
 8805,
 62851,
 17998,
 41013,
 13974,
 51320,
 65533,
 42928,
 4811,
 15185,
 20607,
 60079,
 11844,
 56399,
 48432,
 19309,
 39015,
 7106,
 53531,
 16756,
 64781,
 60748,
 17486,
 4169,
 24335,
 10401,
 21329,
 59156,
 56262,
 49842,
 26024,
 58497,
 46561,
 62629,
 59053,
 21488,
 58400,
 51880,
 65490,
 44219,
 5050,
 16996,
 2730,
 16752,
 26168,
 13361,
 31136,
 64392,
 55574,
 42934]